<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/09%20-%20Motor%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>.

# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining

Neste notebook implementamos um **Motor de Inferência Híbrido** — Forward Chaining (guiado por dados) e Backward Chaining (guiado por metas) — para a base de conhecimento da **Estação de Reabastecimento de Hidrogênio**, reaproveitando as 14 regras de diagnóstico definidas na Aula 08.

- **Forward Chaining:** parte dos fatos de campo (leituras de sensores) e dispara todas as regras aplicáveis até atingir um ponto fixo — útil para saber, em tempo real, tudo que pode ser inferido a partir do estado atual da planta.
- **Backward Chaining:** parte de uma hipótese/meta (ex: "o Tanque de Alta Pressão está em TRIP?") e busca recursivamente, regra por regra, se essa meta pode ser provada a partir dos fatos conhecidos — útil para diagnóstico dirigido, quando o operador quer confirmar uma suspeita específica.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import Set, Tuple, List, Dict, Optional, Any

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r: str, antecedentes: List[str], consequente: str, desc: str, prioridade: int = 1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

    def regras_que_concluem(self, meta: str) -> List[RegraProducao]:
        return [r for r in self.regras if r.consequente == meta]


class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimento):
        self.bc = base_conhecimento

    # ------------------------------------------------------------------
    # FORWARD CHAINING (Data-Driven)
    # ------------------------------------------------------------------
    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        novos_fatos = True
        passo = 1

        while novos_fatos:
            novos_fatos = False
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)
            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "SE": " AND ".join(sorted(regra.antecedentes)),
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    passo += 1
                    novos_fatos = True
                    break
        return fatos_conhecidos, historico_disparos

    # ------------------------------------------------------------------
    # BACKWARD CHAINING (Goal-Driven)
    # ------------------------------------------------------------------
    def backward_chaining(
        self, meta: str, fatos_conhecidos: Set[str],
        _visitados: Optional[Set[str]] = None, _profundidade: int = 0
    ) -> Tuple[bool, List[Dict[str, Any]]]:
        """Tenta provar 'meta' recursivamente, retornando (provado?, trilha_auditoria)."""
        if _visitados is None:
            _visitados = set()

        # Caso base: a meta já é um fato conhecido (leitura direta de sensor/comando)
        if meta in fatos_conhecidos:
            return True, [{
                "Nível": _profundidade, "Meta": meta, "Regra": "-",
                "Resultado": "FATO CONHECIDO", "Detalhe": "Confirmado diretamente pelo campo"
            }]

        # Proteção contra ciclos (meta já sendo investigada nesta mesma cadeia)
        if meta in _visitados:
            return False, [{
                "Nível": _profundidade, "Meta": meta, "Regra": "-",
                "Resultado": "CICLO DETECTADO", "Detalhe": "Meta já em investigação nesta cadeia"
            }]
        _visitados = _visitados | {meta}

        candidatas = self.bc.regras_que_concluem(meta)
        if not candidatas:
            return False, [{
                "Nível": _profundidade, "Meta": meta, "Regra": "-",
                "Resultado": "NÃO PROVADO", "Detalhe": "Nenhuma regra conclui esta meta e não é fato conhecido"
            }]

        for regra in candidatas:
            trilha_regra = []
            todos_provados = True
            for antecedente in sorted(regra.antecedentes):
                provado, sub_trilha = self.backward_chaining(
                    antecedente, fatos_conhecidos, _visitados, _profundidade + 1
                )
                trilha_regra.extend(sub_trilha)
                if not provado:
                    todos_provados = False
                    break
            if todos_provados:
                trilha_regra.append({
                    "Nível": _profundidade, "Meta": meta, "Regra": regra.id_regra,
                    "Resultado": "PROVADO", "Detalhe": regra.descricao_diagnostico
                })
                return True, trilha_regra

        return False, [{
            "Nível": _profundidade, "Meta": meta, "Regra": "-",
            "Resultado": "NÃO PROVADO", "Detalhe": "Nenhuma regra candidata teve todos os antecedentes provados"
        }]


# ============================================================
# BASE DE CONHECIMENTO: ESTAÇÃO DE REABASTECIMENTO DE H₂
# (mesmas 14 regras da Aula 08, no formato simplificado de produção)
# ============================================================

bc = BaseConhecimento()

# --- Setor 100: Armazenamento ---
bc.adicionar_regra("R-01", ["p1_1"], "SOBREPRESSAO_TANQUE_BAIXA", "Pressão do Tanque de Baixa Pressão acima do limite (PT-101 > 400 bar)", 9)
bc.adicionar_regra("R-02", ["SOBREPRESSAO_TANQUE_BAIXA", "v1_1"], "TRIP_TANQUE_BAIXA", "Corte de Segurança do Tanque de Baixa Pressão", 10)
bc.adicionar_regra("R-03", ["g1_1"], "FUGA_H2_TANQUE_BAIXA", "Vazamento de H₂ na área do Tanque de Baixa Pressão (AT-101)", 10)

bc.adicionar_regra("R-04", ["p1_2"], "SOBREPRESSAO_TANQUE_MEDIA", "Pressão do Tanque de Média Pressão acima do limite (PT-102 > 700 bar)", 9)
bc.adicionar_regra("R-05", ["SOBREPRESSAO_TANQUE_MEDIA", "v1_2"], "TRIP_TANQUE_MEDIA", "Corte de Segurança do Tanque de Média Pressão", 10)
bc.adicionar_regra("R-06", ["g1_2"], "FUGA_H2_TANQUE_MEDIA", "Vazamento de H₂ na área do Tanque de Média Pressão (AT-102)", 10)

bc.adicionar_regra("R-07", ["p1_3"], "SOBREPRESSAO_TANQUE_ALTA", "Pressão do Tanque de Alta Pressão acima do limite (PT-103 > 1000 bar)", 9)
bc.adicionar_regra("R-08", ["SOBREPRESSAO_TANQUE_ALTA", "v1_3"], "TRIP_TANQUE_ALTA", "Corte de Segurança do Tanque de Alta Pressão", 10)
bc.adicionar_regra("R-09", ["g1_3"], "FUGA_H2_TANQUE_ALTA", "Vazamento de H₂ na área do Tanque de Alta Pressão (AT-103)", 10)

bc.adicionar_regra("R-10", ["e1_1"], "PARADA_EMERGENCIA_GERAL", "Parada de Emergência acionada manualmente pelo operador (ESD-100)", 10)

# --- Setor 200: Condicionamento ---
bc.adicionar_regra("R-11", ["nc_201", "h3_1"], "BLOQUEIO_DISPENSACAO_TEMPERATURA", "Início de abastecimento solicitado com pré-resfriamento fora da faixa (TT-201 > -40°C)", 6)

# --- Setor 300: Dispensação ---
bc.adicionar_regra("R-12", ["g3_1"], "FUGA_H2_DISPENSADOR", "Vazamento de H₂ detectado na área do dispensador (AT-301)", 10)
bc.adicionar_regra("R-13", ["t3_1"], "SOBRETEMPERATURA_RECEPCAO_VEICULO", "Temperatura no ponto de recepção do veículo acima do limite (TT-301 > 85°C)", 8)
bc.adicionar_regra("R-14", ["p3_1"], "ABASTECIMENTO_CONCLUIDO", "Pressão de enchimento atinge o setpoint do veículo (PT-301 ≈ 700 bar)", 2)

motor = MotorInferencia(bc)

# ============================================================
# DEMONSTRAÇÃO 1: FORWARD CHAINING
# Cenário: Tanque de Alta Pressão com sobrepressão e válvula aberta
# ============================================================
fatos_campo = {"p1_3", "v1_3"}
fatos_finais, trilha_forward = motor.forward_chaining(fatos_campo)

print("=== FORWARD CHAINING — Cenário: PT-103 > 1000 bar, XV-103 ABERTA ===")
print(formatar_tabela(trilha_forward))
print("\nFatos conhecidos ao final:", sorted(fatos_finais))

assert "SOBREPRESSAO_TANQUE_ALTA" in fatos_finais
assert "TRIP_TANQUE_ALTA" in fatos_finais
print("\n[OK] Forward Chaining encadeou corretamente até o TRIP_TANQUE_ALTA!")

# ============================================================
# DEMONSTRAÇÃO 2: BACKWARD CHAINING (meta provável)
# Pergunta do operador: "O TRIP_TANQUE_MEDIA está ativo?"
# ============================================================
fatos_campo_2 = {"p1_2", "v1_2"}
provado, trilha_backward = motor.backward_chaining("TRIP_TANQUE_MEDIA", fatos_campo_2)

print("\n\n=== BACKWARD CHAINING — Meta: TRIP_TANQUE_MEDIA ===")
print(formatar_tabela(trilha_backward))
print(f"\nMeta provada? {provado}")
assert provado is True

# ============================================================
# DEMONSTRAÇÃO 3: BACKWARD CHAINING (meta não provável)
# Pergunta do operador: "Houve FUGA_H2_TANQUE_ALTA?" (sem o fato g1_3 no campo)
# ============================================================
fatos_campo_3 = {"p1_1"}  # nenhuma leitura de vazamento presente
provado_3, trilha_backward_3 = motor.backward_chaining("FUGA_H2_TANQUE_ALTA", fatos_campo_3)

print("\n\n=== BACKWARD CHAINING — Meta: FUGA_H2_TANQUE_ALTA (sem evidência) ===")
print(formatar_tabela(trilha_backward_3))
print(f"\nMeta provada? {provado_3}")
assert provado_3 is False

print("\n[OK] Motor Híbrido de Inferência (Forward + Backward Chaining) validado com sucesso!")


=== FORWARD CHAINING — Cenário: PT-103 > 1000 bar, XV-103 ABERTA ===
Passo | Regra | SE                                | Fato Inferido            | Diagnóstico                                                          
------+-------+-----------------------------------+--------------------------+----------------------------------------------------------------------
1     | R-07  | p1_3                              | SOBREPRESSAO_TANQUE_ALTA | Pressão do Tanque de Alta Pressão acima do limite (PT-103 > 1000 bar)
2     | R-08  | SOBREPRESSAO_TANQUE_ALTA AND v1_3 | TRIP_TANQUE_ALTA         | Corte de Segurança do Tanque de Alta Pressão                         

Fatos conhecidos ao final: ['SOBREPRESSAO_TANQUE_ALTA', 'TRIP_TANQUE_ALTA', 'p1_3', 'v1_3']

[OK] Forward Chaining encadeou corretamente até o TRIP_TANQUE_ALTA!


=== BACKWARD CHAINING — Meta: TRIP_TANQUE_MEDIA ===
Nível | Meta                      | Regra | Resultado      | Detalhe                                                   